# Notebook 10 — Final Three-Model Comparison

Compares **VibrationTransformer**, **VibrationGAT**, and **WAE-GAN + XGBoost** on the held-out test split of the EGB-250 bearing dataset.

| Model | Checkpoint (Google Drive) | Critério de seleção |
|---|---|---|
| VibrationTransformer | `transformers_runs_5_1/runset_20260427_103253/transformer_run_14.pt` | Run 14 |
| VibrationGAT | `gat_runs_20/runset_20260415_180647/gat_run_07.pt` | Menor `best_val_loss` (run 07, seed=48) |
| WAE-GAN + XGBoost | `wae_gan_runs_20/runset_20260415_203901/wae_gan_run_14.pt` + `wae_gan_xgboost_diagnoser_run_14.pkl` | Diagnoser XGBoost pré-salvo (sem re-treino) |

**Constraints:**
- Test split only for evaluation (seed=42, runs 13/4/7 per class)
- Fixed random seeds throughout
- Artefacts salvos em `results/comparison/` dentro do Drive

In [ ]:
# ── Instalação de dependências (rodar apenas uma vez no Colab) ─────────────────
import importlib, subprocess, sys

def _install(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

_install('torch-geometric', 'torch_geometric')
_install('xgboost')

# torch_geometric pode precisar de torch-scatter/sparse em ambientes sem CUDA pré-compilado
try:
    import torch_geometric  # noqa
except ImportError:
    import torch
    torch_ver = torch.__version__.split('+')[0]
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch-geometric',
        '--extra-index-url', f'https://data.pyg.org/whl/torch-{torch_ver}+cpu.html',
    ])

print('Dependências OK ✓')

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import json
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.loader import DataLoader as PyGDataLoader
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Google Colab: montar Drive ────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Project root — Drive no Colab ou diretório local
if IN_COLAB:
    ROOT = Path('/content/drive/MyDrive/anomaly_detection_comparison')
else:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

RESULTS_DIR = ROOT / 'results' / 'comparison'
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['Normal (P1)', 'Inner Race (P2)', 'Roller (P3)', 'Outer Race (P4)']
CLASS_LABELS = [0, 1, 2, 3]
MODEL_COLORS = {'Transformer': '#2196F3', 'GAT': '#4CAF50', 'WAE-GAN+XGB': '#FF5722'}
MODEL_STYLES = {'Transformer': '-', 'GAT': '--', 'WAE-GAN+XGB': ':'}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Colab: {IN_COLAB}  |  Device: {DEVICE}')
print(f'ROOT: {ROOT}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
# ── Cell 2: Checkpoints ────────────────────────────────────────────────────────
# Caminhos hardcoded conforme estrutura no Google Drive
TRANSFORMER_CHECKPOINT = ROOT / 'models/transformers_runs_5_1/runset_20260427_103253/transformer_run_14.pt'
GAT_CHECKPOINT         = ROOT / 'models/gat_runs_20/runset_20260415_180647/gat_run_07.pt'
WAEGAN_CHECKPOINT      = ROOT / 'models/wae_gan_runs_20/runset_20260415_203901/wae_gan_run_14.pt'
WAEGAN_XGB_PATH        = ROOT / 'models/wae_gan_runs_20/runset_20260415_203901/wae_gan_xgboost_diagnoser_run_14.pkl'

# Metadados da run GAT selecionada (de gat_runs_summary.csv)
GAT_RUN_NUM  = 7
GAT_SEED     = 48
GAT_VAL_LOSS = 0.001391

checkpoint_table = pd.DataFrame([
    {'Model': 'Transformer', 'Checkpoint': TRANSFORMER_CHECKPOINT.name,
     'Run': 'run_14', 'Val Loss': '—', 'Seed': '—'},
    {'Model': 'GAT', 'Checkpoint': GAT_CHECKPOINT.name,
     'Run': f'Run {GAT_RUN_NUM}', 'Val Loss': f'{GAT_VAL_LOSS:.6f}', 'Seed': GAT_SEED},
    {'Model': 'WAE-GAN+XGB', 'Checkpoint': WAEGAN_CHECKPOINT.name,
     'Run': 'run_14', 'Val Loss': '—', 'Seed': '—'},
])
print('Selected checkpoints:')
display(checkpoint_table)

for path in [TRANSFORMER_CHECKPOINT, GAT_CHECKPOINT, WAEGAN_CHECKPOINT, WAEGAN_XGB_PATH]:
    assert path.exists(), f'Checkpoint não encontrado: {path}'
print('All checkpoints verified ✓')

In [ ]:
# ── Cell 3: Load Data ─────────────────────────────────────────────────────────
from src.preprocessing import (
    compute_statistical_features,
    build_knn_graph,
    to_sequence_last,
    load_scaler,
    apply_scaler,
)

processed_dir = ROOT / 'data' / 'processed'

X_test  = np.load(processed_dir / 'X_test.npy')   # (N_test, 9, 4096)
y_test  = np.load(processed_dir / 'y_test.npy')   # (N_test,)
X_train = np.load(processed_dir / 'X_train.npy')  # (N_train, 9, 4096)
y_train = np.load(processed_dir / 'y_train.npy')  # (N_train,)

with open(processed_dir / 'split_config.json') as f:
    split_config = json.load(f)

n_classes            = len(np.unique(y_test))
n_test_runs_per_class = len(split_config['test'])  # 3
windows_per_run      = len(X_test) // (n_classes * n_test_runs_per_class)

print(f'X_test  shape: {X_test.shape}  |  classes: {np.unique(y_test, return_counts=True)}')
print(f'X_train shape: {X_train.shape}')
print(f'Test runs per class: {n_test_runs_per_class}  |  windows per run: {windows_per_run}')

# ── Transformer: raw (N, 9, 4096) tensors directly
X_test_t  = torch.from_numpy(X_test.astype(np.float32))
y_test_t  = torch.from_numpy(y_test.astype(np.int64))

# ── GAT: compute 54-d statistical features → k-NN graphs
print('Computing statistical features for GAT...')
X_test_feats = compute_statistical_features(X_test)    # (N_test, 54)
print(f'  Feature shape: {X_test_feats.shape}')
print('Building k-NN graphs...')
graph_test = build_knn_graph(X_test_feats, k=8)        # List[Data]
for g, lbl in zip(graph_test, y_test):
    g.y = torch.tensor(int(lbl), dtype=torch.long)
print(f'  Graphs built: {len(graph_test)}')

# ── WAE-GAN: sequence-last format (N, W, C)
X_test_seq  = to_sequence_last(X_test)   # (N_test, 4096, 9)
X_train_seq = to_sequence_last(X_train)  # (N_train, 4096, 9)

print('Data loading complete ✓')

In [ ]:
# ── Cell 4: Load Models ────────────────────────────────────────────────────────
from src.models.transformer import VibrationTransformer
from src.models.gat import VibrationGAT
from src.models.wae_gan import WAEGAN

# ── 4a: VibrationTransformer ─────────────────────────────────────────────────
trans_ckpt = torch.load(TRANSFORMER_CHECKPOINT, map_location=DEVICE, weights_only=True)
# Infer architecture from checkpoint weights
trans_n_channels = trans_ckpt['input_proj.weight'].shape[-1]
trans_d_model    = trans_ckpt['input_proj.weight'].shape[0]
trans_window     = X_test.shape[-1]  # 4096
trans_num_layers = sum(
    1 for k in trans_ckpt
    if k.startswith('encoder.layers.') and k.endswith('.self_attn.in_proj_weight')
)
trans_dim_ff = trans_ckpt['encoder.layers.0.linear1.weight'].shape[0]
# nhead is not stored directly; use spec defaults (d_model=128→nhead=8, d_model=64→nhead=4)
trans_nhead = 8 if trans_d_model == 128 else 4

transformer = VibrationTransformer(
    n_channels=trans_n_channels,
    window_size=trans_window,
    n_classes=n_classes,
    d_model=trans_d_model,
    nhead=trans_nhead,
    num_layers=trans_num_layers,
    dim_feedforward=trans_dim_ff,
)
transformer.load_state_dict(trans_ckpt)
transformer.to(DEVICE).eval()
n_params_trans = sum(p.numel() for p in transformer.parameters())
print(f'Transformer: n_channels={trans_n_channels}, d_model={trans_d_model}, '
      f'nhead={trans_nhead}, layers={trans_num_layers}, dim_ff={trans_dim_ff}')
print(f'  Parameters: {n_params_trans:,}')

# ── 4b: VibrationGAT ─────────────────────────────────────────────────────────
gat_ckpt = torch.load(GAT_CHECKPOINT, map_location=DEVICE, weights_only=True)

# Infer n_feat from first conv weight (last dim = in_channels)
gat_n_feat = None
for k, v in gat_ckpt.items():
    if 'convs.0' in k and 'weight' in k:
        gat_n_feat = v.shape[-1]
        print(f'GAT n_feat inferred from {k}: shape={tuple(v.shape)} → n_feat={gat_n_feat}')
        break
if gat_n_feat is None:
    gat_n_feat = X_test_feats.shape[-1]
    print(f'GAT n_feat fallback to feature dim: {gat_n_feat}')

# Infer hidden from classifier and num_layers from conv indices
gat_hidden = gat_ckpt['classifier.weight'].shape[-1]
conv_indices = {int(k.split('.')[1]) for k in gat_ckpt if k.startswith('convs.')}
gat_num_layers = max(conv_indices) + 1

# Infer heads: att_src has shape (1, heads, out_channels) in PyG GATConv
gat_heads = 8  # spec default
if 'convs.0.att_src' in gat_ckpt:
    gat_heads = gat_ckpt['convs.0.att_src'].shape[1]
    print(f'GAT heads inferred from att_src: {gat_heads}')

gat = VibrationGAT(
    n_feat=gat_n_feat,
    n_classes=n_classes,
    hidden=gat_hidden,
    heads=gat_heads,
    num_layers=gat_num_layers,
)
gat.load_state_dict(gat_ckpt)
gat.to(DEVICE).eval()
n_params_gat = sum(p.numel() for p in gat.parameters())
print(f'GAT: n_feat={gat_n_feat}, hidden={gat_hidden}, heads={gat_heads}, layers={gat_num_layers}')
print(f'  Parameters: {n_params_gat:,}')

# ── 4c: WAE-GAN ───────────────────────────────────────────────────────────────
waegan = WAEGAN.load(WAEGAN_CHECKPOINT, map_location=str(DEVICE))
waegan.model.eval()
n_params_wae = sum(p.numel() for p in waegan.model.parameters())
print(f'WAE-GAN: embedding_dim={waegan.config.embedding_dim}, '
      f'n_features={waegan.config.n_features}')
print(f'  Parameters: {n_params_wae:,}')

print('\nAll models loaded ✓')

In [ ]:
# ── Cell 5: WAE-GAN Embeddings + XGBoost ──────────────────────────────────────
# Carrega o diagnoser XGBoost pré-salvo (sem re-treino)
import joblib

xgb = joblib.load(WAEGAN_XGB_PATH)
print(f'XGBoost diagnoser carregado: {WAEGAN_XGB_PATH.name}')
print(f'  Classes: {xgb.classes_}  |  n_estimators: {xgb.n_estimators}')

def encode_waegan(waegan_model, X_seq, batch_size=256):
    """Extract mean-pooled encoder embeddings: (N, W, C) → (N, embedding_dim)."""
    waegan_model.model.eval()
    loader = waegan_model.make_dataloader(X_seq, batch_size=batch_size, shuffle=False)
    parts = []
    with torch.no_grad():
        for batch in loader:
            x = batch[0].to(waegan_model.device)
            z = waegan_model.model.encoder(x)   # (B, T, embedding_dim)
            z_pooled = z.mean(dim=1)             # (B, embedding_dim)
            parts.append(z_pooled.cpu().numpy())
    return np.concatenate(parts, axis=0)

print('Codificando conjunto de teste pelo encoder WAE-GAN...')
t0 = time.perf_counter()
test_embeddings = encode_waegan(waegan, X_test_seq)
print(f'  Test embeddings: {test_embeddings.shape}  ({time.perf_counter()-t0:.1f}s)')

wae_y_pred = xgb.predict(test_embeddings)
wae_y_prob = xgb.predict_proba(test_embeddings)   # (N_test, 4)

# Binary anomaly scores (reconstruction error) for separate ROC
print('Computando anomaly scores WAE-GAN...')
wae_anomaly_scores = waegan.predict_anomaly_score(data=X_test_seq)   # (N_test,)

# IQR threshold calibrated on normal-class training windows
normal_mask = y_train == 0
wae_train_scores_normal = waegan.predict_anomaly_score(data=X_train_seq[normal_mask])
wae_threshold = waegan.calculate_threshold(wae_train_scores_normal, multiplier=1.5)

print(f'  Anomaly threshold (IQR): {wae_threshold:.6f}')
print('WAE-GAN setup complete ✓')

In [ ]:
# ── Cell 6: Transformer + GAT Inference ───────────────────────────────────────
from src.evaluation import (
    run_inference_transformer,
    run_inference_gat,
    compute_metrics,
    compute_anomaly_metrics,
    dtw_consistency,
    build_comparison_table,
)

BATCH_SIZE = 256

print('Running Transformer inference...')
trans_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False
)
trans_result = run_inference_transformer(transformer, trans_loader, DEVICE, return_timing=True)
print(f'  Accuracy: {(trans_result.y_pred == trans_result.y_true).mean():.4f} | '
      f'Latency: {trans_result.latency_per_sample_ms:.3f} ms/sample')

print('Running GAT inference...')
gat_loader = PyGDataLoader(graph_test, batch_size=BATCH_SIZE, shuffle=False)
gat_result = run_inference_gat(gat, gat_loader, DEVICE, return_timing=True)
print(f'  Accuracy: {(gat_result.y_pred == gat_result.y_true).mean():.4f} | '
      f'Latency: {gat_result.latency_per_sample_ms:.3f} ms/sample')

# WAE-GAN+XGB timing (excluding embedding step)
t0 = time.perf_counter()
_ = xgb.predict(test_embeddings)
wae_latency_ms = (time.perf_counter() - t0) * 1000 / len(X_test)
print(f'WAE-GAN+XGB (XGB-only) latency: {wae_latency_ms:.4f} ms/sample')

print('Inference complete ✓')

In [ ]:
# ── Cell 7: Compute Metrics ────────────────────────────────────────────────────
trans_metrics = compute_metrics(trans_result.y_true, trans_result.y_pred, trans_result.y_prob)
gat_metrics   = compute_metrics(gat_result.y_true,   gat_result.y_pred,   gat_result.y_prob)
wae_metrics   = compute_metrics(y_test,               wae_y_pred,          wae_y_prob)

# Binary anomaly metrics (WAE-GAN reconstruction error vs normal/anomaly)
y_binary = (y_test != 0).astype(int)
wae_anomaly_metrics = compute_anomaly_metrics(y_binary, wae_anomaly_scores, threshold=wae_threshold)

print('=== WAE-GAN Binary Anomaly Detection ===')
print(f"  AUC-ROC: {wae_anomaly_metrics['auc_roc']:.4f}")
print(f"  Avg Precision: {wae_anomaly_metrics['average_precision']:.4f}")
print(f"  F1 @ threshold: {wae_anomaly_metrics.get('f1', 'n/a')}")

In [ ]:
# ── Cell 8: Side-by-Side Metrics Table ────────────────────────────────────────
rows = [
    {
        'Model': 'Transformer',
        'Accuracy': trans_metrics['accuracy'],
        'Precision (macro)': trans_metrics['precision_macro'],
        'Recall (macro)': trans_metrics['recall_macro'],
        'F1 (macro)': trans_metrics['f1_macro'],
        'AUC-ROC (macro)': trans_metrics['auc_roc_macro'],
        'Latency (ms/sample)': trans_result.latency_per_sample_ms,
        '# Params': n_params_trans,
    },
    {
        'Model': 'GAT',
        'Accuracy': gat_metrics['accuracy'],
        'Precision (macro)': gat_metrics['precision_macro'],
        'Recall (macro)': gat_metrics['recall_macro'],
        'F1 (macro)': gat_metrics['f1_macro'],
        'AUC-ROC (macro)': gat_metrics['auc_roc_macro'],
        'Latency (ms/sample)': gat_result.latency_per_sample_ms,
        '# Params': n_params_gat,
    },
    {
        'Model': 'WAE-GAN+XGB',
        'Accuracy': wae_metrics['accuracy'],
        'Precision (macro)': wae_metrics['precision_macro'],
        'Recall (macro)': wae_metrics['recall_macro'],
        'F1 (macro)': wae_metrics['f1_macro'],
        'AUC-ROC (macro)': wae_metrics['auc_roc_macro'],
        'Latency (ms/sample)': wae_latency_ms,
        '# Params': n_params_wae,
    },
]

comparison_df = pd.DataFrame(rows).set_index('Model')

# Format numeric columns
fmt_pct  = {c: '{:.4f}' for c in ['Accuracy','Precision (macro)','Recall (macro)','F1 (macro)','AUC-ROC (macro)']}
fmt_lat  = {'Latency (ms/sample)': '{:.4f}'}
fmt_par  = {'# Params': '{:,}'}

display(
    comparison_df.style
    .format({**fmt_pct, **fmt_lat})
    .format({'# Params': '{:,.0f}'})
    .highlight_max(axis=0, subset=['Accuracy','F1 (macro)','AUC-ROC (macro)'], color='#c8e6c9')
    .highlight_min(axis=0, subset=['Latency (ms/sample)'], color='#c8e6c9')
    .set_caption('Test-set metrics — best value per column highlighted')
)

comparison_df.to_csv(RESULTS_DIR / 'metrics_table.csv')
print(f"Saved: {RESULTS_DIR / 'metrics_table.csv'}")

In [ ]:
# ── Cell 9: Per-Class Metrics Table ───────────────────────────────────────────
per_class_rows = []
for model_name, metrics in [
    ('Transformer', trans_metrics),
    ('GAT', gat_metrics),
    ('WAE-GAN+XGB', wae_metrics),
]:
    for c, cname in enumerate(CLASS_NAMES):
        per_class_rows.append({
            'Model': model_name,
            'Class': cname,
            'Precision': metrics[f'precision_c{c}'],
            'Recall': metrics[f'recall_c{c}'],
            'F1': metrics[f'f1_c{c}'],
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_pivot = per_class_df.pivot_table(index='Class', columns='Model', values='F1')
display(per_class_pivot.style.format('{:.4f}').highlight_max(axis=1, color='#c8e6c9').set_caption('F1 per class'))
per_class_df.to_csv(RESULTS_DIR / 'per_class_metrics.csv', index=False)
print(f"Saved: {RESULTS_DIR / 'per_class_metrics.csv'}")

In [ ]:
# ── Cell 10: Confusion Matrices ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

model_preds = [
    ('Transformer', trans_result.y_true, trans_result.y_pred),
    ('GAT',         gat_result.y_true,   gat_result.y_pred),
    ('WAE-GAN+XGB', y_test,              wae_y_pred),
]

for ax, (name, y_t, y_p) in zip(axes, model_preds):
    cm = confusion_matrix(y_t, y_p, labels=CLASS_LABELS)
    sns.heatmap(
        cm, annot=True, fmt='d', ax=ax,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cmap='Blues', linewidths=0.5, linecolor='gray',
    )
    acc = (y_t == y_p).mean()
    ax.set_title(f'{name}  (Acc={acc:.4f})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True', fontsize=10)
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out_path = FIGURES_DIR / 'confusion_matrices.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ── Cell 11: ROC Curves (One-vs-Rest, combined) ────────────────────────────────
y_bin = label_binarize(y_test, classes=CLASS_LABELS)  # (N, 4)

model_probs = [
    ('Transformer', trans_result.y_prob),
    ('GAT',         gat_result.y_prob),
    ('WAE-GAN+XGB', wae_y_prob),
]

fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for c, (ax, cname) in enumerate(zip(axes[:4], CLASS_NAMES)):
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, lw=1)
    for mname, y_prob in model_probs:
        fpr, tpr, _ = roc_curve(y_bin[:, c], y_prob[:, c])
        auc = roc_auc_score(y_bin[:, c], y_prob[:, c])
        ax.plot(fpr, tpr,
                color=MODEL_COLORS[mname],
                ls=MODEL_STYLES[mname],
                lw=2,
                label=f'{mname} (AUC={auc:.3f})')
    ax.set_title(cname, fontsize=10)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=7)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

# Macro AUC subplot
ax_macro = axes[4]
ax_macro.plot([0, 1], [0, 1], 'k--', alpha=0.4, lw=1)
for mname, y_prob in model_probs:
    macro_auc = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
    # Plot mean FPR/TPR
    fprs, tprs = [], []
    mean_fpr = np.linspace(0, 1, 200)
    for c in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, c], y_prob[:, c])
        tprs.append(np.interp(mean_fpr, fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0)
    ax_macro.plot(mean_fpr, mean_tpr,
                  color=MODEL_COLORS[mname],
                  ls=MODEL_STYLES[mname],
                  lw=2,
                  label=f'{mname} (AUC={macro_auc:.3f})')
ax_macro.set_title('Macro Average', fontsize=10)
ax_macro.set_xlabel('FPR'); ax_macro.set_ylabel('TPR')
ax_macro.legend(fontsize=7)
ax_macro.set_xlim(0, 1); ax_macro.set_ylim(0, 1.02)

plt.suptitle('ROC Curves (One-vs-Rest)', fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = FIGURES_DIR / 'roc_curves.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ── Cell 12: Precision-Recall Curves (combined) ────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for c, (ax, cname) in enumerate(zip(axes[:4], CLASS_NAMES)):
    for mname, y_prob in model_probs:
        ap = average_precision_score(y_bin[:, c], y_prob[:, c])
        prec, rec, _ = precision_recall_curve(y_bin[:, c], y_prob[:, c])
        ax.plot(rec, prec,
                color=MODEL_COLORS[mname],
                ls=MODEL_STYLES[mname],
                lw=2,
                label=f'{mname} (AP={ap:.3f})')
    # Baseline: class prevalence
    prevalence = y_bin[:, c].mean()
    ax.axhline(prevalence, color='k', ls=':', alpha=0.4, lw=1)
    ax.set_title(cname, fontsize=10)
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(fontsize=7)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

# Macro AP subplot
ax_macro = axes[4]
bars_data = []
for mname, y_prob in model_probs:
    macro_ap = average_precision_score(y_bin, y_prob, average='macro')
    bars_data.append((mname, macro_ap))
    ax_macro.bar(mname, macro_ap, color=MODEL_COLORS[mname], alpha=0.85)
ax_macro.set_ylim(0, 1.0)
ax_macro.set_title('Macro Avg Precision', fontsize=10)
ax_macro.set_ylabel('Average Precision')
for i, (mname, ap) in enumerate(bars_data):
    ax_macro.text(i, ap + 0.01, f'{ap:.3f}', ha='center', fontsize=9)

plt.suptitle('Precision-Recall Curves (One-vs-Rest)', fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = FIGURES_DIR / 'pr_curves.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ── Cell 13: WAE-GAN Binary Anomaly ROC ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))

fpr_a, tpr_a, thresholds_a = roc_curve(y_binary, wae_anomaly_scores)
auc_a = roc_auc_score(y_binary, wae_anomaly_scores)
ax.plot(fpr_a, tpr_a, color=MODEL_COLORS['WAE-GAN+XGB'], lw=2.5,
        label=f'WAE-GAN Reconstruction Error (AUC={auc_a:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, lw=1, label='Random')

# Mark the IQR threshold operating point
idx = np.argmin(np.abs(thresholds_a - wae_threshold))
ax.scatter(fpr_a[idx], tpr_a[idx], s=120, zorder=5,
           color=MODEL_COLORS['WAE-GAN+XGB'], edgecolors='black',
           label=f'IQR threshold (FPR={fpr_a[idx]:.3f}, TPR={tpr_a[idx]:.3f})')

ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('WAE-GAN Binary Anomaly Detection ROC\n(Normal vs Any Fault)', fontweight='bold')
ax.legend(fontsize=9); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
plt.tight_layout()
out_path = FIGURES_DIR / 'wae_anomaly_roc.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ── Cell 14: Error Analysis ────────────────────────────────────────────────────
t_pred = trans_result.y_pred
g_pred = gat_result.y_pred
w_pred = wae_y_pred

# Disagreement: any two models differ
disagreement = (t_pred != g_pred) | (t_pred != w_pred) | (g_pred != w_pred)
all_agree     = ~disagreement

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: disagreement rate per true class
ax = axes[0]
disagree_rates = []
for c, cname in enumerate(CLASS_NAMES):
    mask = y_test == c
    rate = disagreement[mask].mean() if mask.any() else 0.0
    disagree_rates.append(rate)

bars = ax.bar(CLASS_NAMES, disagree_rates, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'], alpha=0.85)
for bar, rate in zip(bars, disagree_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f'{rate:.3f}', ha='center', fontsize=9)
ax.set_title('Model Disagreement Rate per True Class', fontweight='bold')
ax.set_ylabel('Fraction of windows where ≥2 models disagree')
ax.set_ylim(0, min(1.0, max(disagree_rates) * 1.3 + 0.05))
ax.tick_params(axis='x', rotation=20)

# ── Right: who disagrees with whom?
ax2 = axes[1]
pairs = [
    ('Trans≠GAT',     (t_pred != g_pred).sum()),
    ('Trans≠WAE',     (t_pred != w_pred).sum()),
    ('GAT≠WAE',       (g_pred != w_pred).sum()),
    ('All disagree',  ((t_pred != g_pred) & (t_pred != w_pred) & (g_pred != w_pred)).sum()),
    ('All agree',     all_agree.sum()),
]
labels_, counts_ = zip(*pairs)
colors_ = ['#2196F3', '#FF5722', '#4CAF50', '#9E9E9E', '#FFC107']
bars2 = ax2.bar(labels_, counts_, color=colors_, alpha=0.85)
for bar, cnt in zip(bars2, counts_):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             str(int(cnt)), ha='center', fontsize=9)
ax2.set_title('Pairwise Disagreement Counts', fontweight='bold')
ax2.set_ylabel('Number of windows')
ax2.tick_params(axis='x', rotation=15)

plt.suptitle(f'Error Analysis  (total windows={len(y_test)}, '
             f'any disagreement={disagreement.sum()} = {disagreement.mean():.2%})',
             fontweight='bold')
plt.tight_layout()
out_path = FIGURES_DIR / 'error_analysis.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

# Print top disagreement cases
error_df = pd.DataFrame({
    'true_label': y_test,
    'trans_pred': t_pred, 'gat_pred': g_pred, 'wae_pred': w_pred,
    'disagree': disagreement,
})
print(f'\nDisagreement summary (n={disagreement.sum()}):')
print(error_df[error_df.disagree].groupby(['true_label','trans_pred','gat_pred','wae_pred']).size()
      .sort_values(ascending=False).head(10).to_string())

In [ ]:
# ── Cell 15: DTW Temporal Consistency ─────────────────────────────────────────
print(f'Computing DTW consistency (windows_per_run={windows_per_run})...')

trans_dtw = dtw_consistency(trans_result.y_true, trans_result.y_pred, windows_per_run)
gat_dtw   = dtw_consistency(gat_result.y_true,   gat_result.y_pred,   windows_per_run)
wae_dtw   = dtw_consistency(y_test,               wae_y_pred,          windows_per_run)

dtw_results = {'Transformer': trans_dtw, 'GAT': gat_dtw, 'WAE-GAN+XGB': wae_dtw}

fig, ax = plt.subplots(figsize=(7, 4))
models_sorted = list(dtw_results.keys())
dtw_vals = [dtw_results[m] for m in models_sorted]
bars = ax.bar(models_sorted, dtw_vals,
              color=[MODEL_COLORS[m] for m in models_sorted], alpha=0.85)
for bar, val in zip(bars, dtw_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('DTW Temporal Consistency\n(mean DTW distance per run — lower is better)',
             fontweight='bold')
ax.set_ylabel('Mean DTW distance')
ax.set_ylim(0, max(dtw_vals) * 1.25)
plt.tight_layout()
out_path = FIGURES_DIR / 'dtw_consistency.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')
print(f'DTW  — Transformer: {trans_dtw:.4f} | GAT: {gat_dtw:.4f} | WAE-GAN+XGB: {wae_dtw:.4f}')

In [ ]:
# ── Cell 16: Final Ranking ─────────────────────────────────────────────────────
# Weighted score: 40% F1 + 30% AUC-ROC + 20% (inverted DTW) + 10% (inverted latency)
WEIGHTS = {'f1': 0.40, 'auc': 0.30, 'dtw': 0.20, 'latency': 0.10}

metrics_raw = {
    'Transformer': {
        'f1': trans_metrics['f1_macro'],
        'auc': trans_metrics['auc_roc_macro'],
        'dtw': trans_dtw,
        'latency': trans_result.latency_per_sample_ms,
    },
    'GAT': {
        'f1': gat_metrics['f1_macro'],
        'auc': gat_metrics['auc_roc_macro'],
        'dtw': gat_dtw,
        'latency': gat_result.latency_per_sample_ms,
    },
    'WAE-GAN+XGB': {
        'f1': wae_metrics['f1_macro'],
        'auc': wae_metrics['auc_roc_macro'],
        'dtw': wae_dtw,
        'latency': wae_latency_ms,
    },
}

# Min-max normalise each metric; for DTW and latency lower is better → invert
def _normalize(vals):
    lo, hi = min(vals), max(vals)
    return [(v - lo) / (hi - lo) if hi > lo else 0.5 for v in vals]

models = list(metrics_raw.keys())
f1_norm      = _normalize([metrics_raw[m]['f1']      for m in models])
auc_norm     = _normalize([metrics_raw[m]['auc']     for m in models])
dtw_inv_norm = [1 - v for v in _normalize([metrics_raw[m]['dtw']     for m in models])]
lat_inv_norm = [1 - v for v in _normalize([metrics_raw[m]['latency'] for m in models])]

scores = [
    WEIGHTS['f1'] * f + WEIGHTS['auc'] * a + WEIGHTS['dtw'] * d + WEIGHTS['latency'] * l
    for f, a, d, l in zip(f1_norm, auc_norm, dtw_inv_norm, lat_inv_norm)
]

ranking_df = pd.DataFrame({
    'Model': models,
    'F1 macro': [metrics_raw[m]['f1'] for m in models],
    'AUC-ROC': [metrics_raw[m]['auc'] for m in models],
    'DTW': [metrics_raw[m]['dtw'] for m in models],
    'Latency (ms)': [metrics_raw[m]['latency'] for m in models],
    'Weighted Score': scores,
}).sort_values('Weighted Score', ascending=False).reset_index(drop=True)
ranking_df.index += 1

print('\n=== FINAL RANKING ===')
display(
    ranking_df.style
    .format({'F1 macro': '{:.4f}', 'AUC-ROC': '{:.4f}',
             'DTW': '{:.2f}', 'Latency (ms)': '{:.4f}',
             'Weighted Score': '{:.4f}'})
    .highlight_max(subset=['Weighted Score'], color='#a5d6a7')
    .set_caption('Rank 1 = best overall  |  weights: F1=40%, AUC=30%, DTW=20%, latency=10%')
)

winner = ranking_df.iloc[0]['Model']
print(f'\n▶ Best model: {winner}')

In [ ]:
# ── Cell 17: Save Final Report ─────────────────────────────────────────────────
now_str = datetime.now().strftime('%Y-%m-%d %H:%M')

# Build per-class F1 markdown table
pc_md_rows = ['| Class | Transformer F1 | GAT F1 | WAE-GAN+XGB F1 |',
              '|---|---|---|---|']
for c, cname in enumerate(CLASS_NAMES):
    pc_md_rows.append(
        f'| {cname} '
        f'| {trans_metrics[f"f1_c{c}"]:.4f} '
        f'| {gat_metrics[f"f1_c{c}"]:.4f} '
        f'| {wae_metrics[f"f1_c{c}"]:.4f} |'
    )
per_class_md = '\n'.join(pc_md_rows)

rank_md = ranking_df.to_markdown(floatfmt='.4f', index=True)
comp_md = comparison_df.to_markdown(floatfmt='.4f')

report = f"""# Final Comparison Report — EGB-250 Bearing Fault Detection
Generated: {now_str}  |  Device: {DEVICE}  |  RANDOM_SEED={RANDOM_SEED}

---

## 1. Checkpoints Used

| Model | Checkpoint | Selection Criterion |
|---|---|---|
| Transformer | `{TRANSFORMER_CHECKPOINT.name}` | Run 14 (transformers_runs_5_1) |
| GAT | `{GAT_CHECKPOINT.name}` | Menor val_loss em gat_runs_summary.csv (Run {GAT_RUN_NUM}, seed={GAT_SEED}, val_loss={GAT_VAL_LOSS:.6f}) |
| WAE-GAN+XGB | `{WAEGAN_CHECKPOINT.name}` | Diagnoser pré-salvo: `{WAEGAN_XGB_PATH.name}` |

**Test split:** runs {{13, 4, 7}} por classe (seed=42)  
**WAE-GAN anomaly threshold:** {wae_threshold:.6f} (IQR × 1.5 em janelas normais do treino)

---

## 2. Macro Metrics (Test Set)

{comp_md}

---

## 3. Per-Class F1 Score

{per_class_md}

---

## 4. WAE-GAN Binary Anomaly Detection

| Metric | Value |
|---|---|
| AUC-ROC (normal vs any fault) | {wae_anomaly_metrics['auc_roc']:.4f} |
| Average Precision | {wae_anomaly_metrics['average_precision']:.4f} |
| Threshold (IQR-based) | {wae_threshold:.6f} |

---

## 5. Temporal Consistency (DTW)

Mean DTW distance per run (lower = more temporally consistent):

| Model | Mean DTW |
|---|---|
| Transformer | {trans_dtw:.4f} |
| GAT | {gat_dtw:.4f} |
| WAE-GAN+XGB | {wae_dtw:.4f} |

---

## 6. Error Analysis

- Total test windows: {len(y_test)}
- Windows with any model disagreement: {disagreement.sum()} ({disagreement.mean():.2%})
- All models agree: {all_agree.sum()} ({all_agree.mean():.2%})

---

## 7. Final Ranking

Weighted score: **40% F1 + 30% AUC-ROC + 20% inv(DTW) + 10% inv(Latency)**

{rank_md}

---

## 8. Recommendation

**Recommended model for production: {winner}**

### Rationale

- **VibrationTransformer** é um classificador supervisionado de 4 classes que opera diretamente nas janelas brutas. Não requer construção de grafo na inferência, facilitando o deployment. Custo: maior número de parâmetros e latência por amostra em CPU.
- **VibrationGAT** atinge acurácia competitiva com menos parâmetros e inferência mais rápida, ao custo de uma etapa de construção do grafo k-NN na inferência. Ideal para deployment em edge/embedded com restrição de memória.
- **WAE-GAN + XGBoost** oferece tanto scoring de anomalia não-supervisionado (erro de reconstrução) quanto identificação de falha multi-classe (via embeddings → XGBoost). Útil para detectar tipos de falha não vistos no treino. A latência reportada cobre apenas a inferência XGBoost; a codificação pelo encoder WAE-GAN adiciona overhead.

---

## 9. Figures Generated

| File | Description |
|---|---|
| `figures/confusion_matrices.png` | 3-panel confusion matrices (test set) |
| `figures/roc_curves.png` | One-vs-Rest ROC, all 3 models, 4 classes + macro |
| `figures/pr_curves.png` | Precision-Recall curves, all 3 models |
| `figures/wae_anomaly_roc.png` | WAE-GAN binary anomaly ROC with IQR threshold marker |
| `figures/error_analysis.png` | Disagreement rate per class + pairwise counts |
| `figures/dtw_consistency.png` | Mean DTW per model |
"""

report_path = RESULTS_DIR / 'final_report.md'
report_path.write_text(report, encoding='utf-8')
print(f'Report saved: {report_path}')

print('\n=== All artefacts saved under results/comparison/ ===')
for p in sorted(RESULTS_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(RESULTS_DIR)}')